In [ ]:
base    = 'baseline'
current = 'CR'
variant = 'VAR-MS'
cluster = '33'
resolution = '2H'
horizon = [2025, 2030, 2035, 2040, 2045, 2050]
countries_of_interest = ['GB','IT','FR','DK','NL','ES','DE','PL','BE','RO','BG','CZ','NO']
save_fig = True

---
---
### $\text{Load the results}$ 
---
---

In [ ]:
import pandas as pd
import pypsa
import numpy
import pickle
import numpy as np
pd.set_option('display.max_rows', None)
import yaml

In [ ]:
network_bl = {}
network_cr = {}
network_vr = {}

path_bl = f'results/{base}/'
path_cr = f'results/RFNBO_{current}/'
path_vr = f'results/RFNBO_{variant}/'

path_dt_bl = f'results/{base}_vs_RFNBO_{variant}/'
path_dt_cr = f'results/RFNBO_{current}_vs_RFNBO_{variant}/'

for i in horizon:
    network_bl[i] = pypsa.Network(f'{path_bl}networks/base_s_{cluster}__{resolution}_{i}.nc')
    network_bl[i].buses.loc[network_bl[i].buses.index.str.contains('EU'),'country'] = 'EU'

    network_cr[i] = pypsa.Network(f'{path_cr}networks/base_s_{cluster}__{resolution}_{i}.nc')
    network_cr[i].buses.loc[network_cr[i].buses.index.str.contains('EU'),'country'] = 'EU'

    network_vr[i] = pypsa.Network(f'{path_vr}networks/base_s_{cluster}__{resolution}_{i}.nc')
    network_vr[i].buses.loc[network_vr[i].buses.index.str.contains('EU'),'country'] = 'EU'

with open(f"config/config.{base}.yaml", "r") as f:
    config_bl = yaml.safe_load(f)
with open(f"config/config.RFNBO_{current}.yaml", "r") as f:
    config_cr = yaml.safe_load(f)
with open(f"config/config.RFNBO_{variant}.yaml", "r") as f:
    config_vr = yaml.safe_load(f)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

colors = [
    '#C0392B',  # dark red
    '#E67E22',  # burnt orange
    '#B7950B',  # dark mustard
    '#2E7D32',  # forest green
    '#16A085',  # teal
    '#2980B9',  # blue
    '#2C3E50',  # dark slate navy
    '#8E44AD',  # purple
    '#C2185B',  # dark magenta/rose
    '#6D4C41',  # brown
    '#7F8C8D',  # slate gray
    '#00838F',  # dark cyan
    '#2E6E62',  # blue-green
    '#4527A0',  # deep indigo
]

import os
# if not os.path.exists(f'{path_bl}final/') and save_fig:
#     os.mkdir(f'{path_bl}final/')
# if not os.path.exists(f'{path_cr}final/') and save_fig:
#     os.mkdir(f'{path_cr}final/')
if not os.path.exists(f'{path_vr}final/') and save_fig:
    os.mkdir(f'{path_vr}final/')

# if not os.path.exists(f'{path_dt_bl}final/') and save_fig:
#     os.mkdir(f'{path_dt_bl}final/')
# if not os.path.exists(f'{path_dt_cr}final/') and save_fig:
#     os.mkdir(f'{path_dt_cr}final/')


---
---
### $\text{Routines and helpers}$ 
---
---

In [ ]:
def get_total_investment_cost(n):
    """
    Compute total investment (capital) cost for a single PyPSA-Eur network.

    Parameters
    ----------
    n : pypsa.Network

    Returns
    -------
    investment_cost_per_country : dict
        {country_code: total_capital_cost}
    investment_cost_total : float
        Sum of capital costs across all components in the network.
    """
    components_with_investment = {
        "Generator": n.generators,
        "Link": n.links,
        "Store": n.stores,
        "StorageUnit": n.storage_units,
        "Line": n.lines,
    }

    # column names differ slightly by component (p_nom_opt vs e_nom_opt)
    opt_capacity_col = {
        "Generator": "p_nom_opt",
        "Link": "p_nom_opt",
        "Store": "e_nom_opt",
        "StorageUnit": "p_nom_opt",
        "Line": "s_nom_opt",
    }

    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    investment_cost_per_country = {country: 0.0 for country in countries_list}
    investment_cost_total = 0.0

    for comp_name, df in components_with_investment.items():
        if df.empty or "capital_cost" not in df.columns:
            continue

        cap_col = opt_capacity_col[comp_name]
        if cap_col not in df.columns:
            continue

        capital_cost_series = df[cap_col] * df["capital_cost"]

        # attribute each asset's cost to a country based on its bus prefix
        if "bus" in df.columns:
            bus_ref = df["bus"]
        elif "bus0" in df.columns:
            bus_ref = df["bus0"]
        else:
            bus_ref = None

        for idx, cost in capital_cost_series.items():
            if pd.isna(cost):
                continue
            investment_cost_total += cost

            if bus_ref is not None:
                bus_name = bus_ref.loc[idx]
                country_code = bus_name[:2]
                if country_code in investment_cost_per_country:
                    investment_cost_per_country[country_code] += cost

    return investment_cost_per_country, investment_cost_total

In [ ]:
def get_total_investment_cost_vre(n, tech_list_vre=None):
    """
    Compute total investment (capital) cost for a single PyPSA-Eur network,
    restricted to a given list of technologies (by `carrier`).

    Parameters
    ----------
    n : pypsa.Network
    tech_list_vre : list of str, optional
        List of carrier/technology names to include. If None, defaults to
        the VRE tech list.

    Returns
    -------
    investment_cost_per_country : dict
        {country_code: total_capital_cost}
    investment_cost_total : float
        Sum of capital costs across all components in the network,
        restricted to `tech_list_vre`.
    """
    if tech_list_vre is None:
        tech_list_vre = [
            'onwind', 'solar-hsat', 'offwind-float', 'offwind-dc',
            'solar', 'offwind-ac', 'ror', 'solar rooftop',
        ]

    components_with_investment = {
        "Generator": n.generators,
        "Link": n.links,
        "Store": n.stores,
        "StorageUnit": n.storage_units,
        "Line": n.lines,
    }

    # column names differ slightly by component (p_nom_opt vs e_nom_opt)
    opt_capacity_col = {
        "Generator": "p_nom_opt",
        "Link": "p_nom_opt",
        "Store": "e_nom_opt",
        "StorageUnit": "p_nom_opt",
        "Line": "s_nom_opt",
    }

    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    investment_cost_per_country = {country: 0.0 for country in countries_list}
    investment_cost_total = 0.0

    for comp_name, df in components_with_investment.items():
        if df.empty or "capital_cost" not in df.columns:
            continue

        cap_col = opt_capacity_col[comp_name]
        if cap_col not in df.columns:
            continue

        # restrict to the technologies of interest
        if "carrier" not in df.columns:
            continue
        df = df[df["carrier"].isin(tech_list_vre)]
        if df.empty:
            continue

        capital_cost_series = df[cap_col] * df["capital_cost"]

        # attribute each asset's cost to a country based on its bus prefix
        if "bus" in df.columns:
            bus_ref = df["bus"]
        elif "bus0" in df.columns:
            bus_ref = df["bus0"]
        else:
            bus_ref = None

        for idx, cost in capital_cost_series.items():
            if pd.isna(cost):
                continue
            investment_cost_total += cost

            if bus_ref is not None:
                bus_name = bus_ref.loc[idx]
                country_code = bus_name[:2]
                if country_code in investment_cost_per_country:
                    investment_cost_per_country[country_code] += cost

    return investment_cost_per_country, investment_cost_total

In [ ]:
def get_co2(n):
    
    co2_links_bus1 = n.links[
        n.links.bus1.str.contains('co2 atmosphere', case=False, na=False) 
    ]
    co2_links_bus2 = n.links[
        n.links.bus2.str.contains('co2 atmosphere', case=False, na=False) 
    ]
    co2_links_bus3 = n.links[
        n.links.bus3.str.contains('co2 atmosphere', case=False, na=False) 
    ]

    # CO2 from bus1 links
    if not co2_links_bus1.empty:
        co2_bus1 = (
            -n.links_t.p1[co2_links_bus1.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )
    # CO2 from bus2 links
    if not co2_links_bus2.empty:
        co2_bus2 = (
            -n.links_t.p2[co2_links_bus2.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )
    # CO2 from bus3 links
    if not co2_links_bus3.empty:
        co2_bus3 = (
            -n.links_t.p3[co2_links_bus3.index]
            .multiply(n.snapshot_weightings.generators, axis=0)
        )

    co2_all_buses = pd.concat((co2_bus1, co2_bus2, co2_bus3), axis=1).sum(axis = 0)

    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    co2_dict = {country: 0 for country in countries_list}

    for i in co2_all_buses.index:
        co2_dict[i[:2]] += co2_all_buses.loc[i]

    return co2_dict

In [ ]:
def get_electricity(n):
    electricity_buses_names = n.buses[(n.buses.carrier == 'AC')].index

    electricity_prices_per_bus = n.buses_t['marginal_price'][electricity_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    electricity_prices_per_country_dict = {country: [] for country in countries_list}

    electricity_prices_per_bus_dict = dict(electricity_prices_per_bus)

    for i in electricity_prices_per_bus.index:
        electricity_prices_per_country_dict[i[:2]].append(electricity_prices_per_bus.loc[i])
    for key, item in electricity_prices_per_country_dict.items():
        electricity_prices_per_country_dict[key] = np.mean(item)

    return electricity_prices_per_country_dict, electricity_prices_per_bus_dict

In [ ]:
def _get_duals_from_gc(co2_rows, countries_list):
    co2_prices_per_country = {ct: [] for ct in countries_list}

    for name, row in co2_rows.iterrows():
        ct = name[-2:]
        if ct in co2_prices_per_country:
            co2_prices_per_country[ct].append(row["mu"])

    co2_prices_per_country = {
        ct: np.mean(vals) * -1 if vals else np.nan
        for ct, vals in co2_prices_per_country.items()
    }
    return co2_prices_per_country


def _get_duals_from_model(n, countries_list):
    co2_prices_per_country = {ct: [] for ct in countries_list}

    for ct in countries_list:
        ct_duals = []
        for snapshot in n.snapshots:
            constraint_name = f"GlobalConstraint-co2_limit_per_country{ct}"
            try:
                dual_val = n.model.dual[constraint_name].values
                ct_duals.append(float(dual_val))
            except KeyError:
                pass

        co2_prices_per_country[ct] = np.mean(ct_duals) * -1 if ct_duals else np.nan

    return co2_prices_per_country

def get_co2_price(n):
    countries_list = n.buses.country.unique()
    countries_list = countries_list[
        (countries_list != '') & (countries_list != 'EU')
    ]

    gc = n.global_constraints
    co2_rows = gc[gc.index.str.contains("co2_limit_per_country", case=False)]

    # No constraint found → no CO2 price (e.g. 2025 baseline year)
    if co2_rows.empty:
        return {ct: 0.0 for ct in countries_list}

    if co2_rows["mu"].isna().all() or (co2_rows["mu"] == 0).all():
        print("mu column empty — reading duals directly from n.model.dual")
        return _get_duals_from_model(n, countries_list)
    else:
        return _get_duals_from_gc(co2_rows, countries_list)

In [ ]:
def get_hydrogen(n):
    hydrogen_buses_names = n.buses[(n.buses.carrier == 'H2')].index

    hydrogen_prices_per_bus = n.buses_t['marginal_price'][hydrogen_buses_names].mean( axis = 0)
    
    countries_list = n.buses.country.unique()
    countries_list = countries_list[countries_list != '']
    countries_list = countries_list[countries_list != 'EU']

    hydrogen_prices_per_country_dict = {country: [] for country in countries_list}

    hydrogen_prices_per_bus_dict = dict(hydrogen_prices_per_bus)

    for i in hydrogen_prices_per_bus.index:
        hydrogen_prices_per_country_dict[i[:2]].append(hydrogen_prices_per_bus.loc[i])
    for key, item in hydrogen_prices_per_country_dict.items():
        hydrogen_prices_per_country_dict[key] = np.mean(item)

    return hydrogen_prices_per_country_dict, hydrogen_prices_per_bus_dict

In [ ]:
def get_hydrogen_grid_connected(n):

    weights = n.snapshot_weightings.generators

    # 1. Isolate the electrolyzer links
    electrolyzers = n.links[n.links.carrier == 'H2 Electrolysis']
    elec_indices = electrolyzers.index

    # 2. Calculate Annualized Fixed Costs (CAPEX + Fixed O&M) per link
    # capital_cost in PyPSA is already annualized per MW
    fixed_costs = electrolyzers['capital_cost'] * electrolyzers['p_nom_opt']

    # 3. Calculate Variable input costs (Electricity consumed * Local Nodal Price)
    # Extract hourly electricity costs for only those specific buses
    link_to_bus_map = n.links.loc[elec_indices, 'bus0']
    aligned_prices = n.buses_t.marginal_price[link_to_bus_map]
    aligned_prices.columns = elec_indices
    # Extract hourly power consumption at bus0 for each electrolyzer
    hourly_consumption = n.links_t.p0[elec_indices]


    # Multiply element-wise (hourly) and sum over the year to get total variable cost per link
    variable_costs = (hourly_consumption.mul(weights, axis=0) * aligned_prices).sum(axis=0)

    # 4. Calculate total annual hydrogen production per link (Output at bus1)
    # Note: If your model tracks efficiency losses, p1 is the actual H2 generated
    annual_h2_produced = -n.links_t.p1[elec_indices].mul(weights, axis=0).sum(axis=0)

    # 5. Calculate LCOH per individual link
    # Avoid division by zero for links that weren't built (p_nom_opt == 0)
    lcoh_per_link = (fixed_costs + variable_costs) / annual_h2_produced
    lcoh_per_link = lcoh_per_link.dropna()  # Drops links that were not optimized into existence

    lcoh_per_link_capex = fixed_costs / annual_h2_produced
    lcoh_per_link_capex = lcoh_per_link_capex.dropna()  # Drops links that were not optimized into existence
    lcoh_per_link_opex = variable_costs / annual_h2_produced
    lcoh_per_link_opex = lcoh_per_link_opex.dropna()  # Drops links that were not optimized into existence

    # Convert to a clean DataFrame for analysis or plotting
    df_h2 = pd.DataFrame({
        'Bus_Location': electrolyzers.loc[lcoh_per_link.index, 'bus1'],
        'Capacity': electrolyzers.loc[lcoh_per_link.index, 'p_nom_opt'],
        'H2 Produced' : annual_h2_produced,
        'LCOH': lcoh_per_link,
        'LCOH_capex': lcoh_per_link_capex,
        'LCOH_opex': lcoh_per_link_opex
    })

    return df_h2

In [ ]:
def get_vre_share_carbon_intensity_prod(n, config, config_name, year):
    '''
    This function gets the VRE share and co2 intensity (g/kWh) in electricity grid power supply
    from the previous optimised planning horizon which is further used in the additionality constraint.
    Returns a dictionary keyed by country with renewable_share and co2_intensity.
    '''

    params_file = pd.read_csv(
        f"/home/alaterre/pypsa-eur_RFNBO/resources/{config_name}/costs_{year}_processed.csv",
        index_col=[0, 1]
    ).sort_index()
    co2_intensity_raw = params_file["CO2 intensity"]
    co2_intensity_raw.index = co2_intensity_raw.index.droplevel(1)
    co2_intensity_g_kwh = co2_intensity_raw * 1000  # convert t/MWh to g/kWh

    generator_types = list(
        set(config["electricity"]["renewable_carriers"] + ["solar rooftop", "ror"])
    )
    conv_types = list(
        set(config["electricity"]["conventional_carriers"] + [
            "urban central gas CHP", "urban central gas CHP CC",
            "urban central solid biomass CHP", "urban central solid biomass CHP CC",
            "H2 Fuel Cell", "H2 turbine", "geothermal organic rankine cycle"
        ])
    )
    renewable_carriers = generator_types + [
        "urban central solid biomass CHP",
        "urban central solid biomass CHP CC",
        "geothermal organic rankine cycle"
    ]
    gas_chp_types = ["urban central gas CHP", "urban central gas CHP CC"]

    gens = n.generators.index[n.generators.carrier.isin(generator_types)]
    links = n.links.index[n.links.carrier.isin(conv_types)]
    hydro = n.storage_units.index[n.storage_units.carrier == "hydro"]

    countries = [c for c in n.buses.country.unique() if c not in ("EU", "")]

    # --- Weighted production time series (all buses) ---
    gen_p = n.snapshot_weightings.generators @ n.generators_t.p[gens]         # index: bus
    link_p = n.snapshot_weightings.generators @ -n.links_t.p1[links]          # index: bus
    hyd_p = n.snapshot_weightings.generators @ n.storage_units_t.p_dispatch[hydro]  # index: bus

    # --- Weighted fuel consumption time series (for emissions) ---
    gen_consumption = (
        (n.snapshot_weightings.generators @ n.generators_t.p[gens])
        .div(n.generators.loc[gens, "efficiency"])
        .mul(1e3)
    )
    link_consumption = (
        (n.snapshot_weightings.generators @ -n.links_t.p1[links])
        .div(n.links.loc[links, "efficiency"])
        .mul(1e3)
    )

    results = {}

    for country in countries:
        # --- Filter to country ---
        gen_c = (
            gen_p.filter(like=country)
            .groupby(n.generators.loc[gens, "carrier"])
            .sum()
            .mul(1e3)
        )
        link_c = (
            link_p.filter(like=country)
            .groupby(n.links.loc[links, "carrier"])
            .sum()
            .mul(1e3)
        )
        hyd_c = (
            hyd_p.filter(like=country)
            .groupby(n.storage_units.loc[hydro, "carrier"])
            .sum()
            .mul(1e3)
        )

        total_elec = pd.concat([gen_c, link_c, hyd_c])

        # --- CO2 intensity aligned to carriers present ---
        co2_c = co2_intensity_g_kwh.reindex(total_elec.index).fillna(0)
        existing_chp = [t for t in gas_chp_types if t in co2_c.index]
        if existing_chp and "CCGT" in co2_intensity_g_kwh.index:
            co2_c.loc[existing_chp] = co2_intensity_g_kwh.loc["CCGT"]

        # --- Emissions ---
        gen_em = (
            gen_consumption.filter(like=country)
            * n.generators.loc[gen_consumption.filter(like=country).index, "carrier"]
              .map(co2_c)
        )
        gen_em = gen_em.groupby(
            n.generators.loc[gen_em.index, "carrier"]
        ).sum()

        link_em = (
            link_consumption.filter(like=country)
            * n.links.loc[link_consumption.filter(like=country).index, "carrier"]
              .map(co2_c)
        )
        link_em = link_em.groupby(
            n.links.loc[link_em.index, "carrier"]
        ).sum()

        emissions = pd.concat([gen_em, link_em]).groupby(level=0).sum()
        emissions = emissions / 3.6  # convert g/kWh to g/MJ

        # --- Shares ---
        total_generation = total_elec.sum()
        renewable_total = total_elec[total_elec.index.isin(renewable_carriers)].sum()

        results[country] = {
            "renewable_share": 100 * renewable_total / total_generation if total_generation > 0 else 0,
            "co2_intensity": emissions.sum() / total_generation if total_generation > 0 else 0,
        }

    return results


In [ ]:
def get_vre_share_carbon_intensity_cons(n, config, config_name, year):
    '''
    Computes consumption-based renewable share and CO2 intensity per country,
    accounting for cross-border electricity trade (AC lines + DC links).
    
    Steps:
      1. Get production-based renewable share and CO2 intensity per country.
      2. Compute net imports from each trading partner.
      3. Weighted average: self-consumption * domestic intensity + imports * exporter intensity.
    '''

    # --- Step 1: Production-based metrics ---
    prod_based = get_vre_share_carbon_intensity_prod(n, config, config_name, year)
    countries = list(prod_based.keys())

    # --- Step 2: Cross-border net flows (AC lines + DC links) ---
    # net_imports[c_importer][c_exporter] = net MWh imported by c_importer from c_exporter
    net_imports = {c: {c2: 0 for c2 in countries} for c in countries}

    # AC lines
    for line in n.lines.index:
        b0, b1 = n.lines.at[line, "bus0"], n.lines.at[line, "bus1"]
        c0 = n.buses.at[b0, "country"]
        c1 = n.buses.at[b1, "country"]
        if c0 == c1 or c0 not in countries or c1 not in countries:
            continue
        flow = (n.snapshot_weightings.generators @ n.lines_t.p0[line])  # positive = c0 -> c1
        net_imports[c1][c0] += flow
        net_imports[c0][c1] -= flow

    # DC links
    dc_links = n.links.index[n.links.carrier == "DC"]
    for link in dc_links:
        b0, b1 = n.links.at[link, "bus0"], n.links.at[link, "bus1"]
        c0 = n.buses.at[b0, "country"]
        c1 = n.buses.at[b1, "country"]
        if c0 == c1 or c0 not in countries or c1 not in countries:
            continue
        flow = (n.snapshot_weightings.generators @ -n.links_t.p1[link])  # positive = net into c1
        net_imports[c1][c0] += flow
        net_imports[c0][c1] -= flow

    # --- Step 3: Consumption-based weighted average ---
    results = {}

    for c in countries:
        domestic_ren_share = prod_based[c]["renewable_share"] / 100
        domestic_co2 = prod_based[c]["co2_intensity"]

        # Total imports from each partner (only positive = actual net imports)
        total_imports = sum(max(v, 0) for v in net_imports[c].values())

        # Gross production (used to estimate self-consumption)
        # self_consumption = gross_prod - exports = gross_prod - sum of positive outflows
        total_exports = sum(max(-v, 0) for v in net_imports[c].values())

        # Get domestic gross production from production-based function
        # Approximate: use total_elec computed inside prod_based indirectly via
        # renewable_share = ren / total => total = ren / share (if share > 0)
        # Instead, recompute gross production directly here
        generator_types = list(
            set(config["electricity"]["renewable_carriers"] + ["solar rooftop", "ror"])
        )
        conv_types = list(
            set(config["electricity"]["conventional_carriers"] + [
                "urban central gas CHP", "urban central gas CHP CC",
                "urban central solid biomass CHP", "urban central solid biomass CHP CC",
                "H2 Fuel Cell", "H2 turbine", "geothermal organic rankine cycle"
            ])
        )
        gens = n.generators.index[n.generators.carrier.isin(generator_types)]
        links = n.links.index[n.links.carrier.isin(conv_types)]
        hydro = n.storage_units.index[n.storage_units.carrier == "hydro"]

        gross_prod = (
            (n.snapshot_weightings.generators @ n.generators_t.p[gens]).filter(like=c).sum()
            + (n.snapshot_weightings.generators @ -n.links_t.p1[links]).filter(like=c).sum()
            + (n.snapshot_weightings.generators @ n.storage_units_t.p_dispatch[hydro]).filter(like=c).sum()
        )

        self_consumption = gross_prod - total_exports
        total_consumption = self_consumption + total_imports

        if total_consumption <= 0:
            results[c] = {"renewable_share": 0, "co2_intensity": 0}
            continue

        # Weighted sum over self-consumption and each net import source
        weighted_ren = self_consumption * domestic_ren_share
        weighted_co2 = self_consumption * domestic_co2

        for c2, net_imp in net_imports[c].items():
            if net_imp > 0:  # actual net import from c2
                weighted_ren += net_imp * (prod_based[c2]["renewable_share"] / 100)
                weighted_co2 += net_imp * prod_based[c2]["co2_intensity"]

        results[c] = {
            "renewable_share": 100 * weighted_ren / total_consumption,
            "co2_intensity": weighted_co2 / total_consumption,
        }

    return results

---
---
### $\text{Cost of regulation}$ 
---
---

In [ ]:
invest_dict_all_years_bl    = {year : {} for year in horizon}
invest_dict_all_years_cr    = {year : {} for year in horizon}
invest_dict_all_years_vr    = {year : {} for year in horizon}
invest_dict_all_years_dt_bl = {year : {} for year in horizon}
invest_dict_all_years_dt_cr = {year : {} for year in horizon}

total_bl    = {}
total_cr    = {}
total_vr    = {}
total_dt_bl = {}
total_dt_cr = {}

for year in horizon:
    invest_dict_all_years_bl[year], total_bl[year] = get_total_investment_cost(network_bl[year])
    invest_dict_all_years_cr[year], total_cr[year] = get_total_investment_cost(network_cr[year])
    invest_dict_all_years_vr[year], total_vr[year] = get_total_investment_cost(network_vr[year])
    invest_dict_all_years_dt_bl[year], total_dt_bl[year] = (
        pd.Series(invest_dict_all_years_vr[year]) - pd.Series(invest_dict_all_years_bl[year])
    ).to_dict(), total_vr[year]-total_bl[year]
    invest_dict_all_years_dt_cr[year], total_dt_cr[year] = (
        pd.Series(invest_dict_all_years_vr[year]) - pd.Series(invest_dict_all_years_cr[year])
    ).to_dict(), total_vr[year]-total_cr[year]

In [ ]:
invest_vre_dict_all_years_bl    = {year : {} for year in horizon}
invest_vre_dict_all_years_cr    = {year : {} for year in horizon}
invest_vre_dict_all_years_vr    = {year : {} for year in horizon}
invest_vre_dict_all_years_dt_bl = {year : {} for year in horizon}
invest_vre_dict_all_years_dt_cr = {year : {} for year in horizon}

total_vre_bl    = {}
total_vre_cr    = {}
total_vre_vr    = {}
total_vre_dt_bl = {}
total_vre_dt_cr = {}

for year in horizon:
    invest_vre_dict_all_years_bl[year], total_vre_bl[year] = get_total_investment_cost_vre(network_bl[year])
    invest_vre_dict_all_years_cr[year], total_vre_cr[year] = get_total_investment_cost_vre(network_cr[year])
    invest_vre_dict_all_years_vr[year], total_vre_vr[year] = get_total_investment_cost_vre(network_vr[year])
    invest_vre_dict_all_years_dt_bl[year], total_vre_dt_bl[year] = (
        pd.Series(invest_vre_dict_all_years_vr[year]) - pd.Series(invest_vre_dict_all_years_bl[year])
    ).to_dict(), total_vre_vr[year]-total_vre_bl[year]
    invest_vre_dict_all_years_dt_cr[year], total_vre_dt_cr[year] = (
        pd.Series(invest_vre_dict_all_years_vr[year]) - pd.Series(invest_vre_dict_all_years_cr[year])
    ).to_dict(), total_vre_vr[year]-total_vre_cr[year]

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    
    invest_country_dt_bl = np.array([invest_dict[country] for invest_dict in invest_dict_all_years_dt_bl.values()])
    invest_country_bl    = np.array([invest_dict[country] for invest_dict in invest_dict_all_years_bl.values()])
    axs[0,0].plot(horizon, [v for v in invest_country_dt_bl / invest_country_bl * 100],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('Relative Difference [%]', fontsize=12)
axs[0,0].set_title(f'Country Level (= {variant} - {base})', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    
    invest_country_dt_cr = np.array([invest_dict[country] for invest_dict in invest_dict_all_years_dt_cr.values()])
    invest_country_cr    = np.array([invest_dict[country] for invest_dict in invest_dict_all_years_cr.values()])
    axs[0,1].plot(horizon, [v for v in invest_country_dt_cr / invest_country_cr * 100],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,1].set_xlabel('Year', fontsize=12)
axs[0,1].set_ylabel('Relative Difference [%]', fontsize=12)
axs[0,1].set_title(f'Country Level (= {variant} - {current})', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

# --- Bottom plot: EU total ---
invest_total_dt_bl = np.array([invest_dict for invest_dict in total_dt_bl.values()])
invest_total_bl    = np.array([invest_dict for invest_dict in total_bl.values()])

axs[1,0].plot(horizon, [v for v in invest_total_dt_bl / invest_total_bl * 100],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('Relative Difference [%]', fontsize=12)
axs[1,0].set_title(f'EU Total (= {variant} - {base}, cum. = {sum(invest_total_dt_bl) / sum(invest_total_bl) * 1e2:.1f}%)', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

invest_total_dt_cr = np.array([invest_dict for invest_dict in total_dt_cr.values()])
invest_total_cr    = np.array([invest_dict for invest_dict in total_cr.values()])

axs[1,1].plot(horizon, [v for v in invest_total_dt_cr / invest_total_cr * 100],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,1].set_xlabel('Year', fontsize=12)
axs[1,1].set_ylabel('Relative Difference [%]', fontsize=12)
axs[1,1].set_title(f'EU Total (= {variant} - {current}, cum. = {sum(invest_total_dt_cr) / sum(invest_total_cr) * 1e2:.1f}%)', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

title = 'Relative Difference in Total Investment Costs Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    
    invest_country_dt_bl = np.array([invest_vre_dict[country] for invest_vre_dict in invest_vre_dict_all_years_dt_bl.values()])
    invest_country_bl    = np.array([invest_vre_dict[country] for invest_vre_dict in invest_vre_dict_all_years_bl.values()])
    axs[0,0].plot(horizon, [v for v in invest_country_dt_bl / invest_country_bl * 100],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('Relative Difference [%]', fontsize=12)
axs[0,0].set_title(f'Country Level (= {variant} - {base})', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    
    invest_country_dt_cr = np.array([invest_vre_dict[country] for invest_vre_dict in invest_vre_dict_all_years_dt_cr.values()])
    invest_country_cr    = np.array([invest_vre_dict[country] for invest_vre_dict in invest_vre_dict_all_years_cr.values()])
    axs[0,1].plot(horizon, [v for v in invest_country_dt_cr / invest_country_cr * 100],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,1].set_xlabel('Year', fontsize=12)
axs[0,1].set_ylabel('Relative Difference [%]', fontsize=12)
axs[0,1].set_title(f'Country Level (= {variant} - {current})', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

# --- Bottom plot: EU total ---
invest_total_vre_dt_bl = np.array([invest_vre_dict for invest_vre_dict in total_vre_dt_bl.values()])
invest_total_vre_bl    = np.array([invest_vre_dict for invest_vre_dict in total_vre_bl.values()])

axs[1,0].plot(horizon, [v for v in invest_total_vre_dt_bl / invest_total_vre_bl * 100],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('Relative Difference [%]', fontsize=12)
axs[1,0].set_title(f'EU Total (= {variant} - {base}, cum. = {sum(invest_total_vre_dt_bl) / sum(invest_total_vre_bl) * 1e2:.1f}%)', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

invest_total_vre_dt_cr = np.array([invest_vre_dict for invest_vre_dict in total_vre_dt_cr.values()])
invest_total_vre_cr    = np.array([invest_vre_dict for invest_vre_dict in total_vre_cr.values()])

axs[1,1].plot(horizon, [v for v in invest_total_vre_dt_cr / invest_total_vre_cr * 100],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,1].set_xlabel('Year', fontsize=12)
axs[1,1].set_ylabel('Relative Difference [%]', fontsize=12)
axs[1,1].set_title(f'EU Total (= {variant} - {current}, cum. = {sum(invest_total_vre_dt_cr) / sum(invest_total_vre_cr) * 1e2:.1f}%)', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

title = 'Relative Difference in Renewables Investment Costs Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

---
---
### $\text{CO}_2 \text{ emissions and price}$ 
---
---

In [ ]:
co2_dict_all_years_bl    = {year : {} for year in horizon}
co2_dict_all_years_cr    = {year : {} for year in horizon}
co2_dict_all_years_vr    = {year : {} for year in horizon}
co2_dict_all_years_dt_bl = {year : {} for year in horizon}
co2_dict_all_years_dt_cr = {year : {} for year in horizon}

for year in horizon:
    co2_dict_all_years_bl[year] = get_co2(network_bl[year])
    co2_dict_all_years_cr[year] = get_co2(network_cr[year])
    co2_dict_all_years_vr[year] = get_co2(network_vr[year])
    co2_dict_all_years_dt_bl[year] = (pd.Series(co2_dict_all_years_vr[year]) - pd.Series(co2_dict_all_years_bl[year])
    ).to_dict()
    co2_dict_all_years_dt_cr[year] = (pd.Series(co2_dict_all_years_vr[year]) - pd.Series(co2_dict_all_years_cr[year])
    ).to_dict()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(21, 12), sharey='row')

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    co2_country = [co2_dict[country] for co2_dict in co2_dict_all_years_bl.values()]
    axs[0,0].plot(horizon, [v / 1e6 for v in co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
axs[0,0].set_title(f'Country Level ({base} scenario)', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    co2_country = [co2_dict[country] for co2_dict in co2_dict_all_years_cr.values()]
    axs[0,1].plot(horizon, [v / 1e6 for v in co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

axs[0,1].set_xlabel('Year', fontsize=12)
#axs[0,1].set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
axs[0,1].set_title(f'Country Level ({current} scenario)', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    co2_country = [co2_dict[country] for co2_dict in co2_dict_all_years_vr.values()]
    axs[0,2].plot(horizon, [v / 1e6 for v in co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

axs[0,2].set_xlabel('Year', fontsize=12)
#axs[0,2].set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
axs[0,2].set_title(f'Country Level ({variant} scenario)', fontsize=14, fontweight='bold')
axs[0,2].legend(fontsize=11, framealpha=0.9)
axs[0,2].grid(alpha=0.3, linestyle='--')
axs[0,2].set_xticks(horizon)
axs[0,2].spines['top'].set_visible(False)
axs[0,2].spines['right'].set_visible(False)

# --- Bottom plot: EU total ---
co2_EU = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_bl.values()]
axs[1,0].plot(horizon, [v / 1e6 for v in co2_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
axs[1,0].set_title(f'EU Total ({base} scenario)', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

co2_EU = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_cr.values()]
axs[1,1].plot(horizon, [v / 1e6 for v in co2_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,1].set_xlabel('Year', fontsize=12)
#axs[1,1].set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
axs[1,1].set_title(f'EU Total ({current} scenario)', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

co2_EU = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_vr.values()]
axs[1,2].plot(horizon, [v / 1e6 for v in co2_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,2].set_xlabel('Year', fontsize=12)
#axs[1,2].set_ylabel('CO₂ Emissions [MtCO₂]', fontsize=12)
axs[1,2].set_title(f'EU Total ({variant} scenario)', fontsize=14, fontweight='bold')
axs[1,2].legend(fontsize=11, framealpha=0.9)
axs[1,2].grid(alpha=0.3, linestyle='--')
axs[1,2].set_xticks(horizon)
axs[1,2].spines['top'].set_visible(False)
axs[1,2].spines['right'].set_visible(False)

title = 'Net CO₂ Emissions Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    co2_country = [co2_dict[country] for co2_dict in co2_dict_all_years_dt_bl.values()]
    axs[0,0].plot(horizon, [v / 1e6 for v in co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

axs[0,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('Difference in CO₂ Emissions [MtCO₂]', fontsize=12)
axs[0,0].set_title(f'Country Level (= {variant} - {base})', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    co2_country = [co2_dict[country] for co2_dict in co2_dict_all_years_dt_cr.values()]
    axs[0,1].plot(horizon, [v / 1e6 for v in co2_country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

axs[0,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,1].set_xlabel('Year', fontsize=12)
axs[0,1].set_ylabel('Difference in CO₂ Emissions [MtCO₂]', fontsize=12)
axs[0,1].set_title(f'Country Level (= {variant} - {current})', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

# --- Bottom plot: EU total ---
co2_EU_bl = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_bl.values()]
co2_EU_cr = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_cr.values()]
co2_EU_dt_bl = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_dt_bl.values()]
co2_EU_dt_cr = [sum(co2_dict.values())-co2_dict['EU'] for co2_dict in co2_dict_all_years_dt_cr.values()]

axs[1,0].plot(horizon, [v / 1e6 for v in co2_EU_dt_bl],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('Difference in CO₂ Emissions [MtCO₂]', fontsize=12)
axs[1,0].set_title(f'EU Total (= {variant} - {base}, cum. = {sum(co2_EU_dt_bl) / 1e6:.0f} MtCO₂, {sum(co2_EU_dt_bl) / sum(co2_EU_bl) * 1e2:.1f}%)', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

axs[1,1].plot(horizon, [v / 1e6 for v in co2_EU_dt_cr],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,1].set_xlabel('Year', fontsize=12)
axs[1,1].set_ylabel('Difference in CO₂ Emissions [MtCO₂]', fontsize=12)
axs[1,1].set_title(f'EU Total (= {variant} - {current}, cum. = {sum(co2_EU_dt_cr) / 1e6:.0f} MtCO₂, {sum(co2_EU_dt_cr) / sum(co2_EU_cr) * 1e2:.1f}%)', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

title = 'Absolute Difference in CO₂ Emissions Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

---
---
### $\text{Electricity Prices}$ 
---
---

In [ ]:
electricity_per_bus_dict_all_years_bl        = {year : {} for year in horizon}
electricity_per_bus_dict_all_years_cr        = {year : {} for year in horizon}
electricity_per_bus_dict_all_years_vr        = {year : {} for year in horizon}
electricity_per_bus_dict_all_years_dt_bl     = {year : {} for year in horizon}
electricity_per_bus_dict_all_years_dt_cr     = {year : {} for year in horizon}

electricity_per_country_dict_all_years_bl    = {year : {} for year in horizon}
electricity_per_country_dict_all_years_cr    = {year : {} for year in horizon}
electricity_per_country_dict_all_years_vr    = {year : {} for year in horizon}
electricity_per_country_dict_all_years_dt_bl = {year : {} for year in horizon}
electricity_per_country_dict_all_years_dt_cr = {year : {} for year in horizon}

for year in horizon:
    electricity_per_country_dict_all_years_bl[year], electricity_per_bus_dict_all_years_bl[year] = get_electricity(network_bl[year])
    electricity_per_country_dict_all_years_cr[year], electricity_per_bus_dict_all_years_cr[year] = get_electricity(network_cr[year])
    electricity_per_country_dict_all_years_vr[year], electricity_per_bus_dict_all_years_vr[year] = get_electricity(network_vr[year])
    
    electricity_per_country_dict_all_years_dt_bl[year], electricity_per_bus_dict_all_years_dt_bl[year] = (pd.Series(electricity_per_country_dict_all_years_vr[year]) - pd.Series(electricity_per_country_dict_all_years_bl[year])).to_dict(), (pd.Series(electricity_per_bus_dict_all_years_vr[year]) - pd.Series(electricity_per_bus_dict_all_years_bl[year])).to_dict()
    electricity_per_country_dict_all_years_dt_cr[year], electricity_per_bus_dict_all_years_dt_cr[year] = (pd.Series(electricity_per_country_dict_all_years_vr[year]) - pd.Series(electricity_per_country_dict_all_years_cr[year])).to_dict(), (pd.Series(electricity_per_bus_dict_all_years_vr[year]) - pd.Series(electricity_per_bus_dict_all_years_cr[year])).to_dict()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(21, 12), sharey='row')

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    electricity_country = [electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_bl.values()]
    axs[0,0].plot(horizon, [v for v in electricity_country],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('Marginal Electricity Cost [€/MWh]', fontsize=12)
axs[0,0].set_title(f'Country Level ({base} scenario)', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    electricity_country = [electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_cr.values()]
    axs[0,1].plot(horizon, [v for v in electricity_country],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,1].set_xlabel('Year', fontsize=12)
#axs[0,1].set_ylabel('Marginal Electricity Cost [€/MWh]', fontsize=12)
axs[0,1].set_title(f'Country Level ({current} scenario)', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    electricity_country = [electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_vr.values()]
    axs[0,2].plot(horizon, [v for v in electricity_country],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,2].set_xlabel('Year', fontsize=12)
#axs[0,2].set_ylabel('Marginal Electricity Cost [€/MWh]', fontsize=12)
axs[0,2].set_title(f'Country Level ({variant} scenario)', fontsize=14, fontweight='bold')
axs[0,2].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,2].grid(alpha=0.3, linestyle='--')
axs[0,2].set_xticks(horizon)
axs[0,2].spines['top'].set_visible(False)
axs[0,2].spines['right'].set_visible(False)

# --- Bottom plot: EU average ---
electricity_EU = [np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_bl.values()]
axs[1,0].plot(horizon, [v for v in electricity_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('Marginal Electricity Cost [€/MWh]', fontsize=12)
axs[1,0].set_title(f'EU Average ({base} scenario)', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

electricity_EU = [np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_cr.values()]
axs[1,1].plot(horizon, [v for v in electricity_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,1].set_xlabel('Year', fontsize=12)
#axs[1,1].set_ylabel('Marginal Electricity Cost [€/MWh]', fontsize=12)
axs[1,1].set_title(f'EU Average ({current} scenario)', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

electricity_EU = [np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_vr.values()]
axs[1,2].plot(horizon, [v for v in electricity_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,2].set_xlabel('Year', fontsize=12)
#axs[1,2].set_ylabel('Marginal Electricity Cost [€/MWh]', fontsize=12)
axs[1,2].set_title(f'EU Average ({variant} scenario)', fontsize=14, fontweight='bold')
axs[1,2].legend(fontsize=11, framealpha=0.9)
axs[1,2].grid(alpha=0.3, linestyle='--')
axs[1,2].set_xticks(horizon)
axs[1,2].spines['top'].set_visible(False)
axs[1,2].spines['right'].set_visible(False)

title = 'Marginal Electricity Costs Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    electricity_country = np.array([electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_dt_bl.values()])
    axs[0,0].plot(horizon, [v for v in electricity_country],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('Difference in Marginal Electricity Cost [€/MWh]', fontsize=12)
axs[0,0].set_title(f'Country Level (= {variant} - {base})', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    electricity_country = np.array([electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_dt_cr.values()])
    axs[0,1].plot(horizon, [v for v in electricity_country],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,1].set_xlabel('Year', fontsize=12)
axs[0,1].set_ylabel('Difference in Marginal Electricity Cost [€/MWh]', fontsize=12)
axs[0,1].set_title(f'Country Level (= {variant} - {current})', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

# --- Bottom plot: EU total ---
electricity_EU = [np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_dt_bl.values()]
axs[1,0].plot(horizon, [v for v in electricity_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('Difference in Marginal Electricity Cost [€/MWh]', fontsize=12)
axs[1,0].set_title(f'EU Average', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

electricity_EU = [np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_dt_cr.values()]
axs[1,1].plot(horizon, [v for v in electricity_EU],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,1].set_xlabel('Year', fontsize=12)
axs[1,1].set_ylabel('Difference in Marginal Electricity Cost [€/MWh]', fontsize=12)
axs[1,1].set_title(f'EU Average', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

title = 'Absolute Difference in Marginal Electricity Costs Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    electricity_country_dt_bl = np.array([electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_dt_bl.values()])
    electricity_country_bl    = np.array([electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_bl.values()])
    axs[0,0].plot(horizon, electricity_country_dt_bl / electricity_country_bl * 100,
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('Difference in Marginal Electricity Cost [%]', fontsize=12)
axs[0,0].set_title(f'Country Level (= {variant} - {base})', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    electricity_country_dt_cr = np.array([electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_dt_cr.values()])
    electricity_country_cr    = np.array([electricity_dict[country] for electricity_dict in electricity_per_country_dict_all_years_cr.values()])
    axs[0,1].plot(horizon, electricity_country_dt_cr / electricity_country_cr * 100,
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country)

axs[0,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,1].set_xlabel('Year', fontsize=12)
axs[0,1].set_ylabel('Difference in Marginal Electricity Cost [%]', fontsize=12)
axs[0,1].set_title(f'Country Level (= {variant} - {current})', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

# --- Bottom plot: EU average ---
electricity_EU_dt_bl = np.array([np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_dt_bl.values()])
electricity_EU_bl    = np.array([np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_bl.values()])
axs[1,0].plot(horizon, electricity_EU_dt_bl / electricity_EU_bl * 100,
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('Difference in Marginal Electricity Cost [%]', fontsize=12)
axs[1,0].set_title(f'EU Average', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

electricity_EU_dt_cr = np.array([np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_dt_cr.values()])
electricity_EU_cr    = np.array([np.mean(list(electricity_dict.values())) for electricity_dict in electricity_per_bus_dict_all_years_cr.values()])
axs[1,1].plot(horizon, electricity_EU_dt_cr / electricity_EU_cr * 100,
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,1].set_xlabel('Year', fontsize=12)
axs[1,1].set_ylabel('Difference in Marginal Electricity Cost [%]', fontsize=12)
axs[1,1].set_title(f'EU Average', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

title = 'Relative Difference in Marginal Electricity Costs Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

---
---
### $\text{Grid connected hydrogen}$ 
---
---

In [ ]:
h2_produced_all_bl          = {c: [] for c in countries_of_interest}
h2_produced_all_bl['EU']    = []
h2_produced_all_cr          = {c: [] for c in countries_of_interest}
h2_produced_all_cr['EU']    = []
h2_produced_all_vr          = {c: [] for c in countries_of_interest}
h2_produced_all_vr['EU']    = []

lcoh_all_bl          = {c: [] for c in countries_of_interest}
lcoh_all_bl['EU']    = []
lcoh_all_cr          = {c: [] for c in countries_of_interest}
lcoh_all_cr['EU']    = []
lcoh_all_vr          = {c: [] for c in countries_of_interest}
lcoh_all_vr['EU']    = []

lcoh_capex_all_bl          = {c: [] for c in countries_of_interest}
lcoh_capex_all_bl['EU']    = []
lcoh_capex_all_cr          = {c: [] for c in countries_of_interest}
lcoh_capex_all_cr['EU']    = []
lcoh_capex_all_vr          = {c: [] for c in countries_of_interest}
lcoh_capex_all_vr['EU']    = []

lcoh_opex_all_bl          = {c: [] for c in countries_of_interest}
lcoh_opex_all_bl['EU']    = []
lcoh_opex_all_cr          = {c: [] for c in countries_of_interest}
lcoh_opex_all_cr['EU']    = []
lcoh_opex_all_vr          = {c: [] for c in countries_of_interest}
lcoh_opex_all_vr['EU']    = []

for year in horizon:

    df_h2_bl = get_hydrogen_grid_connected(network_bl[year])
    h2_produced_bl = {country: 0  for country in countries_of_interest}
    lcoh_bl        = {country: [] for country in countries_of_interest}
    lcoh_bl_capex  = {country: [] for country in countries_of_interest}
    lcoh_bl_opex   = {country: [] for country in countries_of_interest}

    for i in df_h2_bl.index:
        if i[:2] in h2_produced_bl:
            h2_produced_bl[i[:2]] += df_h2_bl.loc[i, 'H2 Produced']*1e-6
            lcoh_bl[i[:2]].append(df_h2_bl.loc[i, 'LCOH'])
            lcoh_bl_capex[i[:2]].append(df_h2_bl.loc[i, 'LCOH_capex'])
            lcoh_bl_opex[i[:2]].append(df_h2_bl.loc[i, 'LCOH_opex'])

    for key in lcoh_bl:
        lcoh_bl[key] = np.mean(lcoh_bl[key])
        lcoh_bl_capex[key] = np.mean(lcoh_bl_capex[key])
        lcoh_bl_opex[key] = np.mean(lcoh_bl_opex[key])

    h2_produced_bl['EU'] = df_h2_bl['H2 Produced'].sum()*1e-6
    lcoh_bl['EU']        = np.mean(df_h2_bl['LCOH'])
    lcoh_bl_capex['EU']  = np.mean(df_h2_bl['LCOH_capex'])
    lcoh_bl_opex['EU']   = np.mean(df_h2_bl['LCOH_opex'])

    for c in countries_of_interest + ['EU']:
        h2_produced_all_bl[c].append(h2_produced_bl[c])
        lcoh_all_bl[c].append(lcoh_bl[c])
        lcoh_capex_all_bl[c].append(lcoh_bl_capex[c])
        lcoh_opex_all_bl[c].append(lcoh_bl_opex[c])


    df_h2_cr = get_hydrogen_grid_connected(network_cr[year])
    h2_produced_cr = {country: 0  for country in countries_of_interest}
    lcoh_cr        = {country: [] for country in countries_of_interest}
    lcoh_cr_capex  = {country: [] for country in countries_of_interest}
    lcoh_cr_opex   = {country: [] for country in countries_of_interest}

    for i in df_h2_cr.index:
        if i[:2] in h2_produced_cr:
            h2_produced_cr[i[:2]] += df_h2_cr.loc[i, 'H2 Produced']*1e-6
            lcoh_cr[i[:2]].append(df_h2_cr.loc[i, 'LCOH'])
            lcoh_cr_capex[i[:2]].append(df_h2_cr.loc[i, 'LCOH_capex'])
            lcoh_cr_opex[i[:2]].append(df_h2_cr.loc[i, 'LCOH_opex'])

    for key in lcoh_cr:
        lcoh_cr[key] = np.mean(lcoh_cr[key])
        lcoh_cr_capex[key] = np.mean(lcoh_cr_capex[key])
        lcoh_cr_opex[key] = np.mean(lcoh_cr_opex[key])

    h2_produced_cr['EU'] = df_h2_cr['H2 Produced'].sum()*1e-6
    lcoh_cr['EU']        = np.mean(df_h2_cr['LCOH'])
    lcoh_cr_capex['EU']  = np.mean(df_h2_cr['LCOH_capex'])
    lcoh_cr_opex['EU']   = np.mean(df_h2_cr['LCOH_opex'])

    for c in countries_of_interest + ['EU']:
        h2_produced_all_cr[c].append(h2_produced_cr[c])
        lcoh_all_cr[c].append(lcoh_cr[c])
        lcoh_capex_all_cr[c].append(lcoh_cr_capex[c])
        lcoh_opex_all_cr[c].append(lcoh_cr_opex[c])
    

    df_h2_vr = get_hydrogen_grid_connected(network_vr[year])
    h2_produced_vr = {country: 0  for country in countries_of_interest}
    lcoh_vr        = {country: [] for country in countries_of_interest}
    lcoh_vr_capex  = {country: [] for country in countries_of_interest}
    lcoh_vr_opex   = {country: [] for country in countries_of_interest}

    for i in df_h2_vr.index:
        if i[:2] in h2_produced_vr:
            h2_produced_vr[i[:2]] += df_h2_vr.loc[i, 'H2 Produced']*1e-6
            lcoh_vr[i[:2]].append(df_h2_vr.loc[i, 'LCOH'])
            lcoh_vr_capex[i[:2]].append(df_h2_vr.loc[i, 'LCOH_capex'])
            lcoh_vr_opex[i[:2]].append(df_h2_vr.loc[i, 'LCOH_opex'])

    for key in lcoh_vr:
        lcoh_vr[key] = np.mean(lcoh_vr[key])
        lcoh_vr_capex[key] = np.mean(lcoh_vr_capex[key])
        lcoh_vr_opex[key] = np.mean(lcoh_vr_opex[key])

    h2_produced_vr['EU'] = df_h2_vr['H2 Produced'].sum()*1e-6
    lcoh_vr['EU']        = np.mean(df_h2_vr['LCOH'])
    lcoh_vr_capex['EU']  = np.mean(df_h2_vr['LCOH_capex'])
    lcoh_vr_opex['EU']   = np.mean(df_h2_vr['LCOH_opex'])

    for c in countries_of_interest + ['EU']:
        h2_produced_all_vr[c].append(h2_produced_vr[c])
        lcoh_all_vr[c].append(lcoh_vr[c])
        lcoh_capex_all_vr[c].append(lcoh_vr_capex[c])
        lcoh_opex_all_vr[c].append(lcoh_vr_opex[c])


h2_produced_all_dt_bl = {key: np.array(h2_produced_all_vr[key] - np.array(h2_produced_all_bl[key])) for key in h2_produced_all_bl}
lcoh_all_dt_bl        = {key: np.array(lcoh_all_vr[key])       - np.array(lcoh_all_bl[key])         for key in lcoh_all_bl}
lcoh_capex_all_dt_bl  = {key: np.array(lcoh_capex_all_vr[key]) - np.array(lcoh_capex_all_bl[key])   for key in lcoh_capex_all_bl}
lcoh_opex_all_dt_bl   = {key: np.array(lcoh_opex_all_vr[key])  - np.array(lcoh_opex_all_bl[key])    for key in lcoh_opex_all_bl}

h2_produced_all_dt_cr = {key: np.array(h2_produced_all_vr[key] - np.array(h2_produced_all_cr[key])) for key in h2_produced_all_cr}
lcoh_all_dt_cr        = {key: np.array(lcoh_all_vr[key])       - np.array(lcoh_all_cr[key])         for key in lcoh_all_cr}
lcoh_capex_all_dt_cr  = {key: np.array(lcoh_capex_all_vr[key]) - np.array(lcoh_capex_all_cr[key])   for key in lcoh_capex_all_cr}
lcoh_opex_all_dt_cr   = {key: np.array(lcoh_opex_all_vr[key])  - np.array(lcoh_opex_all_cr[key])    for key in lcoh_opex_all_cr}

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(21, 12), sharey='row')

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    axs[0,0].plot(horizon, h2_produced_all_bl[country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('H₂ Produced [TWh]', fontsize=12)
axs[0,0].set_title(f'Country Level ({base} scenario)', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    axs[0,1].plot(horizon, h2_produced_all_cr[country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

axs[0,1].set_xlabel('Year', fontsize=12)
#axs[0,1].set_ylabel('H₂ Produced [TWh]', fontsize=12)
axs[0,1].set_title(f'Country Level ({current} scenario)', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    axs[0,2].plot(horizon, h2_produced_all_vr[country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

axs[0,2].set_xlabel('Year', fontsize=12)
#axs[0,2].set_ylabel('H₂ Produced [TWh]', fontsize=12)
axs[0,2].set_title(f'Country Level ({variant} scenario)', fontsize=14, fontweight='bold')
axs[0,2].legend(fontsize=11, framealpha=0.9)
axs[0,2].grid(alpha=0.3, linestyle='--')
axs[0,2].set_xticks(horizon)
axs[0,2].spines['top'].set_visible(False)
axs[0,2].spines['right'].set_visible(False)

# --- Bottom plot: EU total ---
axs[1,0].plot(horizon, h2_produced_all_bl['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('H₂ Produced [TWh]', fontsize=12)
axs[1,0].set_title(f'EU Total ({base} scenario)', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

axs[1,1].plot(horizon, h2_produced_all_cr['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,1].set_xlabel('Year', fontsize=12)
#axs[1,1].set_ylabel('H₂ Produced [TWh]', fontsize=12)
axs[1,1].set_title(f'EU Total ({current} scenario)', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

axs[1,2].plot(horizon, h2_produced_all_vr['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,2].set_xlabel('Year', fontsize=12)
#axs[1,2].set_ylabel('H₂ Produced [TWh]', fontsize=12)
axs[1,2].set_title(f'EU Total ({variant} scenario)', fontsize=14, fontweight='bold')
axs[1,2].legend(fontsize=11, framealpha=0.9)
axs[1,2].grid(alpha=0.3, linestyle='--')
axs[1,2].set_xticks(horizon)
axs[1,2].spines['top'].set_visible(False)
axs[1,2].spines['right'].set_visible(False)

title = 'Grid Connected H₂ Production Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    axs[0,0].plot(horizon, h2_produced_all_dt_bl[country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

axs[0,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('Difference in H₂ Produced [TWh]', fontsize=12)
axs[0,0].set_title(f'Country Level (= {variant} - {base})', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    axs[0,1].plot(horizon, h2_produced_all_dt_cr[country],
             color=colors[i], lw=2, marker='o', markersize=6,
             label=country)

axs[0,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,1].set_xlabel('Year', fontsize=12)
axs[0,1].set_ylabel('Difference in H₂ Produced [TWh]', fontsize=12)
axs[0,1].set_title(f'Country Level (= {variant} - {current})', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

# --- Bottom plot: EU total ---
axs[1,0].plot(horizon, h2_produced_all_dt_bl['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('Difference in H₂ Produced [TWh]', fontsize=12)
axs[1,0].set_title(f'EU Total (= {variant} - {base}, cum. = {sum(h2_produced_all_dt_bl['EU']) / 1e0:.0f} TWh, {sum(h2_produced_all_dt_bl["EU"])/sum(h2_produced_all_bl["EU"]) * 100:.1f}%)', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

axs[1,1].plot(horizon, h2_produced_all_dt_cr['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU total', linestyle='--')

axs[1,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,1].set_xlabel('Year', fontsize=12)
axs[1,1].set_ylabel('Difference in H₂ Produced [TWh]', fontsize=12)
axs[1,1].set_title(f'EU Total (= {variant} - {current}, cum. = {sum(h2_produced_all_dt_cr['EU']) / 1e0:.0f} TWh, {sum(h2_produced_all_dt_cr["EU"])/sum(h2_produced_all_cr["EU"]) * 100:.1f}%)', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

title = 'Absolute Difference in Grid Connected H₂ Production Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(21, 12), sharey='row')

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    
    if sum(h2_produced_all_bl[country]) > 10:
        axs[0,0].plot(horizon, lcoh_all_bl[country],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country+' (> 10 TWh cum.)')
    else:
        axs[0,0].plot(horizon, lcoh_all_bl[country],
             color=colors[i], lw=1.5, marker='o', markersize=4.5,
             label=country+' (< 10 TWh cum.)',alpha=0.5)

axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('LCOH₂ [€/MWh]', fontsize=12)
axs[0,0].set_title(f'Country Level ({base} scenario)', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):

    if sum(h2_produced_all_cr[country]) > 10:
        axs[0,1].plot(horizon, lcoh_all_cr[country],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country+' (> 10 TWh cum.)')
    else:
        axs[0,1].plot(horizon, lcoh_all_cr[country],
             color=colors[i], lw=1.5, marker='o', markersize=4.5,
             label=country+' (< 10 TWh cum.)',alpha=0.5)

axs[0,1].set_xlabel('Year', fontsize=12)
#axs[0,1].set_ylabel('LCOH₂ [€/MWh]', fontsize=12)
axs[0,1].set_title(f'Country Level ({current} scenario)', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    
    if sum(h2_produced_all_vr[country]) > 10:
        axs[0,2].plot(horizon, lcoh_all_vr[country],
                 color=colors[i], lw=2, marker='o', markersize=6,
                 label=country+' (> 10 TWh cum.)')
    else:
        axs[0,2].plot(horizon, lcoh_all_vr[country],
                 color=colors[i], lw=1.5, marker='o', markersize=4.5,
                 label=country+' (< 10 TWh cum.)',alpha=0.5)

axs[0,2].set_xlabel('Year', fontsize=12)
#axs[0,2].set_ylabel('LCOH₂ [€/MWh]', fontsize=12)
axs[0,2].set_title(f'Country Level ({variant} scenario)', fontsize=14, fontweight='bold')
axs[0,2].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,2].grid(alpha=0.3, linestyle='--')
axs[0,2].set_xticks(horizon)
axs[0,2].spines['top'].set_visible(False)
axs[0,2].spines['right'].set_visible(False)

# --- Bottom plot: EU Average ---
axs[1,0].plot(horizon, lcoh_all_bl['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('LCOH₂ [€/MWh]', fontsize=12)
axs[1,0].set_title(f'EU Average ({base} scenario)', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

axs[1,1].plot(horizon, lcoh_all_cr['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,1].set_xlabel('Year', fontsize=12)
#axs[1,1].set_ylabel('LCOH₂ [€/MWh]', fontsize=12)
axs[1,1].set_title(f'EU Average ({current} scenario)', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

axs[1,2].plot(horizon, lcoh_all_vr['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,2].set_xlabel('Year', fontsize=12)
#axs[1,2].set_ylabel('LCOH₂ [€/MWh]', fontsize=12)
axs[1,2].set_title(f'EU Average ({variant} scenario)', fontsize=14, fontweight='bold')
axs[1,2].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[1,2].grid(alpha=0.3, linestyle='--')
axs[1,2].set_xticks(horizon)
axs[1,2].spines['top'].set_visible(False)
axs[1,2].spines['right'].set_visible(False)

title = 'Levelized Cost of Grid Connected H₂ Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    
    if sum(h2_produced_all_vr[country]) > 10:
        axs[0,0].plot(horizon, lcoh_all_dt_bl[country],
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country+' (> 10 TWh cum.)')
    else:
        axs[0,0].plot(horizon, lcoh_all_dt_bl[country],
             color=colors[i], lw=1.5, marker='o', markersize=4.5,
             label=country+' (< 10 TWh cum.)',alpha=0.5)

axs[0,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('Difference in LCOH₂ [€/MWh]', fontsize=12)
axs[0,0].set_title(f'Country Level (= {variant} - {base})', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    
    if sum(h2_produced_all_vr[country]) > 10:
        axs[0,1].plot(horizon, lcoh_all_dt_cr[country],
                 color=colors[i], lw=2, marker='o', markersize=6,
                 label=country+' (> 10 TWh cum.)')
    else:
        axs[0,1].plot(horizon, lcoh_all_dt_cr[country],
                 color=colors[i], lw=1.5, marker='o', markersize=4.5,
                 label=country+' (< 10 TWh cum.)',alpha=0.5)

axs[0,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,1].set_xlabel('Year', fontsize=12)
axs[0,1].set_ylabel('Difference in LCOH₂ [€/MWh]', fontsize=12)
axs[0,1].set_title(f'Country Level (= {variant} - {current})', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

# --- Bottom plot: EU Average ---
axs[1,0].plot(horizon, lcoh_all_dt_bl['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('Difference in LCOH₂ [€/MWh]', fontsize=12)
axs[1,0].set_title(f'EU Average (= {variant} - {base})', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

axs[1,1].plot(horizon, lcoh_all_dt_cr['EU'],
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,1].set_xlabel('Year', fontsize=12)
axs[1,1].set_ylabel('Difference in LCOH₂ [€/MWh]', fontsize=12)
axs[1,1].set_title(f'EU Average (= {variant} - {current})', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

title = 'Absolute Difference in Levelized Cost of Grid Connected H₂ Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 12))

# --- Top plot: countries ---
for i, country in enumerate(countries_of_interest):
    
    if sum(h2_produced_all_vr[country]) > 10:
        axs[0,0].plot(horizon, np.array(lcoh_all_dt_bl[country])/np.array(lcoh_all_bl[country]) * 100,
                color=colors[i], lw=2, marker='o', markersize=6,
                label=country+' (> 10 TWh cum.)')
    else:
        axs[0,0].plot(horizon, np.array(lcoh_all_dt_bl[country])/np.array(lcoh_all_bl[country]) * 100,
             color=colors[i], lw=1.5, marker='o', markersize=4.5,
             label=country+' (< 10 TWh cum.)',alpha=0.5)

axs[0,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,0].set_xlabel('Year', fontsize=12)
axs[0,0].set_ylabel('Relative Difference in LCOH₂ [%]', fontsize=12)
axs[0,0].set_title(f'Country Level (= {variant} - {base})', fontsize=14, fontweight='bold')
axs[0,0].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,0].grid(alpha=0.3, linestyle='--')
axs[0,0].set_xticks(horizon)
axs[0,0].spines['top'].set_visible(False)
axs[0,0].spines['right'].set_visible(False)

for i, country in enumerate(countries_of_interest):
    
    if sum(h2_produced_all_vr[country]) > 10:
        axs[0,1].plot(horizon, np.array(lcoh_all_dt_cr[country])/np.array(lcoh_all_cr[country]) * 100,
                 color=colors[i], lw=2, marker='o', markersize=6,
                 label=country+' (> 10 TWh cum.)')
    else:
        axs[0,1].plot(horizon, np.array(lcoh_all_dt_cr[country])/np.array(lcoh_all_cr[country]) * 100,
                 color=colors[i], lw=1.5, marker='o', markersize=4.5,
                 label=country+' (< 10 TWh cum.)',alpha=0.5)

axs[0,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[0,1].set_xlabel('Year', fontsize=12)
axs[0,1].set_ylabel('Relative Difference in LCOH₂ [%]', fontsize=12)
axs[0,1].set_title(f'Country Level (= {variant} - {current})', fontsize=14, fontweight='bold')
axs[0,1].legend(fontsize=11, framealpha=0.9, ncols=2)
axs[0,1].grid(alpha=0.3, linestyle='--')
axs[0,1].set_xticks(horizon)
axs[0,1].spines['top'].set_visible(False)
axs[0,1].spines['right'].set_visible(False)

# --- Bottom plot: EU Average ---
axs[1,0].plot(horizon, np.array(lcoh_all_dt_bl['EU'])/np.array(lcoh_all_bl['EU']) * 100,
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,0].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,0].set_xlabel('Year', fontsize=12)
axs[1,0].set_ylabel('Relative Difference in LCOH₂ [%]', fontsize=12)
axs[1,0].set_title(f'EU Average (= {variant} - {base})', fontsize=14, fontweight='bold')
axs[1,0].legend(fontsize=11, framealpha=0.9)
axs[1,0].grid(alpha=0.3, linestyle='--')
axs[1,0].set_xticks(horizon)
axs[1,0].spines['top'].set_visible(False)
axs[1,0].spines['right'].set_visible(False)

axs[1,1].plot(horizon, np.array(lcoh_all_dt_cr['EU'])/np.array(lcoh_all_cr['EU']) * 100,
         color=colors[-1], lw=2.5, marker='o', markersize=6,
         label='EU average', linestyle='--')

axs[1,1].axhline(0, color='darkgrey', linewidth=1.0, linestyle='--', zorder=0)
axs[1,1].set_xlabel('Year', fontsize=12)
axs[1,1].set_ylabel('Relative Difference in LCOH₂ [%]', fontsize=12)
axs[1,1].set_title(f'EU Average (= {variant} - {current})', fontsize=14, fontweight='bold')
axs[1,1].legend(fontsize=11, framealpha=0.9)
axs[1,1].grid(alpha=0.3, linestyle='--')
axs[1,1].set_xticks(horizon)
axs[1,1].spines['top'].set_visible(False)
axs[1,1].spines['right'].set_visible(False)

title = 'Relative Difference in Levelized Cost of Grid Connected H₂ Over Time'
plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
if save_fig: plt.savefig(f'{path_vr}final/{title.replace(" ", "_")}.svg', bbox_inches='tight')
plt.show()